In [76]:
from httpx import ReadTimeout
import nba_api
import pandas as pd
import time
import pprint
import json
import os
from nba_api.stats.static import players

In [41]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.1.7-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchainhub-0.1.21-py3-none-any.whl.metadata (659 bytes)
  Using cached chromadb-1.4.1-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached langchain_core-1.2.7-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached types_requests-2.32.4.20260107-py3-none-any.whl.metadata (2.0 kB)
  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached uvicorn-0.40.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached posthog-5.4.0-py3-none-any.whl.metadata (5.7 kB)
  Using 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 5.29.5 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\natha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_AI_API_KEY")

In [97]:
import sqlite3

# Connect to your local DB
db_path = "data/nba_stats.db"

def get_schema():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # Query to get all table names and their CREATE statements
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
    schema = "\n".join([row[0] for row in cursor.fetchall() if row[0]])
    conn.close()
    return schema

db_schema = get_schema()

In [98]:
import pprint
pprint.pprint(db_schema)
with open('db_schema.txt', 'w') as f:
    f.write(db_schema)

('CREATE TABLE player_season_stats (\n'
 '        player_id INTEGER NOT NULL,\n'
 '        player_name TEXT NOT NULL,\n'
 '        team_id INTEGER,\n'
 '        team_abbreviation TEXT,\n'
 '        season TEXT NOT NULL,\n'
 '        age REAL,\n'
 '        gp INTEGER,\n'
 '        w INTEGER,\n'
 '        l INTEGER,\n'
 '        w_pct REAL,\n'
 '        min REAL,\n'
 '        fgm REAL, fga REAL, fg_pct REAL,\n'
 '        fg3m REAL, fg3a REAL, fg3_pct REAL,\n'
 '        ftm REAL, fta REAL, ft_pct REAL,\n'
 '        oreb REAL, dreb REAL, reb REAL,\n'
 '        ast REAL, tov REAL,\n'
 '        stl REAL, blk REAL, pf REAL,\n'
 '        pts REAL, plus_minus REAL,\n'
 '        ts_pct REAL, efg_pct REAL,\n'
 '        pts_per36 REAL, reb_per36 REAL, ast_per36 REAL,\n'
 '        stl_per36 REAL, blk_per36 REAL,\n'
 '        PRIMARY KEY (player_id, season)\n'
 '    )\n'
 'CREATE TABLE modern_players (player_id INTEGER PRIMARY KEY, full_name TEXT '
 'NOT NULL, is_active INTEGER)\n'
 'CREATE TABLE pl

In [99]:
from pydantic import BaseModel, Field
class SQLQueryResult(BaseModel):
    table_name: str = Field(description="The name of the table used in the query")
    query: str = Field(description="The executable SQLite query")
    file_name: str = Field(description="The name of the file used in the query")

In [136]:
from langchain_core.prompts import ChatPromptTemplate

# Custom prompt for SQL generation
sql_template = """You are a SQLite expert. Given the following database schema, 
write a SQL query that answers the user's question. Your response should ACCURATELY reflect the column NAMES and STRUCTURE in the schema. do NOT deviate from this
Also, use context clues from the question to infer any filters that should be applied. Example; someone asking about "KD" should be filtered to "Kevin Durant" in the player_name column, as that is his real name and should be used
Output ONLY the SQL code, no preamble.
Stats that the user asks about (points, rebounds, assists, etc.) should be in the form of "pts", "reb", "ast", etc. as they are in the database, and should be assumed to be in their base form (per game, not per 36 minutes) unless otherwise specified.
Use outside information to transform any nicknames or abbreviations into the correct values in the database.
Additionally, seasons should be in the format "2023-24" for the 2024 season, "2013-14" for the 2014 season, etc. the ONLY exception being if asked about draft YEAR, if someone asks for players drafted in 2020, the draft_year column should be filtered to 2020, not 2019-20.
Percentage stats should be in decimal form, so 50% should be 0.5, 45.3% should be 0.453, etc.
Counting stats are all in their per game form, if a user asks for totals on the season, multiply those stats by games played
IMPORTANT: Always subtract 0.01 from any counting stats, so if the user asks for a specific counting stats, such as "who averaged >=24 ppg", write in the SQL >=23.99 instead of >=24, to account for any rounding issues in the data. Mind this should only be applied to counting stats like points, rebounds, assists, steals and blocks, not percentage stats like shooting percentages, or advanced stats like PER, win shares, etc. 
Here's the order I'd like you to give me the information
1. The table name that is most relevant to the user's question, based on the schema and the question. Only output one table name, and it should be the most relevant one to the user's question. This will be used to determine which table to query from.
2. The SQL query (self explanatory)
3. The file name for the resulting query output (should match the information you gave, so if you were asked to compare lebron and kd, it would be something like "lebron_kd_comparison")

Schema:
{schema}

Question: {question}

SQL Query:"""

sql_prompt = ChatPromptTemplate.from_template(sql_template)

In [120]:
import json
from pathlib import Path

# ---------- path ----------
DRAFT_JSON_PATH = Path("data/draft_history.json")

def fix_json_seasons():
    if not DRAFT_JSON_PATH.exists():
        print(f"File not found: {DRAFT_JSON_PATH}")
        return

    # 1. Load the existing data
    with open(DRAFT_JSON_PATH, 'r') as f:
        data = json.load(f)

    # 2. Update the season format
    # This converts "2024" -> "2023-24" or "2025" -> "2024-25"
    # Adjust the logic if you specifically want "2024-25" for draft year 2025
    for entry in data:
        raw_season = entry.get("SEASON")
        if raw_season and len(raw_season) == 4:
            year = int(raw_season)
            # Typically, the 2024 Draft corresponds to the 2024-25 season
            # If you want the draft year to match the upcoming season:
            formatted = f"{year}-{str(year + 1)[-2:]}"
            entry["SEASON"] = formatted

    # 3. Save it back to the same file
    with open(DRAFT_JSON_PATH, 'w') as f:
        json.dump(data, f, indent=2)

    print(f"✅ Successfully updated seasons in {DRAFT_JSON_PATH}")

if __name__ == "__main__":
    fix_json_seasons()

✅ Successfully updated seasons in data\draft_history.json


In [137]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model_name="gpt-4o", temperature=0)
structured_llm = llm.with_structured_output(SQLQueryResult)
# The chain: Prompt -> LLM -> String Output
sql_chain = sql_prompt | structured_llm

# Define your question
question= "players who've made 175 total threes in a season since 2023-24 "
# Generate the SQL
generated_sql = sql_chain.invoke({"schema": db_schema, "question": question})
pprint.pprint(f"Generated SQL:\n{generated_sql}")

('Generated SQL:\n'
 'table_name=\'player_season_stats\' query="SELECT player_name, season, fg3m * '
 "gp AS total_threes\\nFROM player_season_stats\\nWHERE season >= '2023-24' "
 'AND (fg3m * gp) >= 174.99;" file_name=\'players_175_threes_since_2023\'')


In [138]:
df = pd.read_csv("data/nba_player_stats_2000_2025.csv")
#bub carrington player id is 1642267
df[df['PLAYER_NAME'] == "Bub Carrington"][['PLAYER_ID', 'PLAYER_NAME']]

,PLAYER_ID,PLAYER_NAME
11725,1642267,Bub Carrington


In [139]:
import pandas as pd

# Execute the query and store in a variable
conn = sqlite3.connect(db_path)
df_result = pd.read_sql_query(generated_sql.query, conn)
print(df_result)
conn.close()

           player_name   season  total_threes
0      Anthony Edwards  2023-24         189.6
1    Bogdan Bogdanovic  2023-24         237.0
2       Brandon Miller  2023-24         185.0
3          Buddy Hield  2023-24         218.4
4          CJ McCollum  2023-24         237.6
5           Coby White  2023-24         205.4
6        Corey Kispert  2023-24         184.0
7     D'Angelo Russell  2023-24         228.0
8       Damian Lillard  2023-24         219.0
9         De'Aaron Fox  2023-24         214.6
10     Dejounte Murray  2023-24         202.8
11       Derrick White  2023-24         197.1
12    Donovan Mitchell  2023-24         181.5
13    Donte DiVincenzo  2023-24         283.5
14     Duncan Robinson  2023-24         190.4
15       Fred VanVleet  2023-24         226.3
16      Gary Trent Jr.  2023-24         177.5
17       Grayson Allen  2023-24         202.5
18       Jalen Brunson  2023-24         207.9
19         Jalen Green  2023-24         205.0
20        James Harden  2023-24   

In [140]:
import sqlite3

def get_real_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # This pulls the actual SQL used to create your tables
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
    schema_details = cursor.fetchall()
    conn.close()
    return "\n".join([row[0] for row in schema_details if row[0]])

actual_schema = get_real_schema("data/nba_stats.db")
print(actual_schema)

CREATE TABLE player_season_stats (
        player_id INTEGER NOT NULL,
        player_name TEXT NOT NULL,
        team_id INTEGER,
        team_abbreviation TEXT,
        season TEXT NOT NULL,
        age REAL,
        gp INTEGER,
        w INTEGER,
        l INTEGER,
        w_pct REAL,
        min REAL,
        fgm REAL, fga REAL, fg_pct REAL,
        fg3m REAL, fg3a REAL, fg3_pct REAL,
        ftm REAL, fta REAL, ft_pct REAL,
        oreb REAL, dreb REAL, reb REAL,
        ast REAL, tov REAL,
        stl REAL, blk REAL, pf REAL,
        pts REAL, plus_minus REAL,
        ts_pct REAL, efg_pct REAL,
        pts_per36 REAL, reb_per36 REAL, ast_per36 REAL,
        stl_per36 REAL, blk_per36 REAL,
        PRIMARY KEY (player_id, season)
    )
CREATE TABLE modern_players (player_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, is_active INTEGER)
CREATE TABLE player_missing_seasons (player_name TEXT, seasons_json TEXT, missing_seasons_json TEXT)
CREATE TABLE draft_history (person_id INTEGER

In [141]:
import json

evidence_packet = {
    "question": question,
    "row_count": len(df_result),
    "data": df_result.to_dict(orient="records")
}

evidence_json = json.dumps(evidence_packet, ensure_ascii=False, indent=2)

print(evidence_json)


{
  "question": "players who've made 175 total threes in a season since 2023-24 ",
  "row_count": 70,
  "data": [
    {
      "player_name": "Anthony Edwards",
      "season": "2023-24",
      "total_threes": 189.6
    },
    {
      "player_name": "Bogdan Bogdanovic",
      "season": "2023-24",
      "total_threes": 237.0
    },
    {
      "player_name": "Brandon Miller",
      "season": "2023-24",
      "total_threes": 185.0
    },
    {
      "player_name": "Buddy Hield",
      "season": "2023-24",
      "total_threes": 218.4
    },
    {
      "player_name": "CJ McCollum",
      "season": "2023-24",
      "total_threes": 237.6
    },
    {
      "player_name": "Coby White",
      "season": "2023-24",
      "total_threes": 205.4
    },
    {
      "player_name": "Corey Kispert",
      "season": "2023-24",
      "total_threes": 184.0
    },
    {
      "player_name": "D'Angelo Russell",
      "season": "2023-24",
      "total_threes": 228.0
    },
    {
      "player_name": "Damian 

In [142]:
import pandas as pd

df = pd.DataFrame(evidence_packet["data"])
pd.set_option("display.max_rows", None)
df

,player_name,season,total_threes
0,Anthony Edwards,2023-24,189.6
1,Bogdan Bogdanovic,2023-24,237.0
2,Brandon Miller,2023-24,185.0
3,Buddy Hield,2023-24,218.4
4,CJ McCollum,2023-24,237.6
5,Coby White,2023-24,205.4
6,Corey Kispert,2023-24,184.0
7,D'Angelo Russell,2023-24,228.0
8,Damian Lillard,2023-24,219.0
9,De'Aaron Fox,2023-24,214.6


In [ ]:
import pickle 
with open("temp_data/" + generated_sql.table_name + ".csv", "wb") as f:
    pickle.dump(evidence_packet, f) 

In [145]:
SYSTEM_RULES = """You are an NBA stats analyst.
You MUST answer using ONLY the evidence provided.
Do NOT use outside knowledge.
Do NOT guess or invent numbers.

Keep the answer factual.
"""

USER_PROMPT = f"""User Question:
{question}

Evidence:
{evidence_json}

Return your answer in EXACTLY this format:

Short Answer: The short, concise answer to the user's question, based ONLY on the evidence provided.
Full Dataset: Answer in the form of a table with appropriate rows and columns, based ONLY on the evidence provided. Pretty much, just format the json file in a readable table format, listing EVERY single row NO SKIPPING (IMPORTANT), don't summarize the data. Do NOT use outside knowledge or guess at any numbers. 
Summary: A comprehensive summary of what the data says in relation to the user's question, based ONLY on the evidence provided. Do NOT use outside knowledge or guess at any numbers.
Source: NBAstats.com

IMPORTANT: Before answering "Not Enough Data", check to make sure that the evidence provided is actually insufficient to answer the question. If the data is relevant but just doesn't directly answer the question, you can still use it to provide a short answer and summary, but just say "Not enough data for a comprehensive answer" or something like that in the summary section. Only say "Not enough data" if there is truly no relevant information in the evidence to even partially answer the question.
"""

llm_answer = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

response = llm_answer.invoke([
    {"role": "system", "content": SYSTEM_RULES},
    {"role": "user", "content": USER_PROMPT}
])

print(response.content)


Short Answer: The players who've made 175 total threes in a season since 2023-24 are Anthony Edwards, Bogdan Bogdanovic, Brandon Miller, Buddy Hield, CJ McCollum, Coby White, Corey Kispert, D'Angelo Russell, Damian Lillard, De'Aaron Fox, Dejounte Murray, Derrick White, Donovan Mitchell, Donte DiVincenzo, Duncan Robinson, Fred VanVleet, Gary Trent Jr., Grayson Allen, Jalen Brunson, Jalen Green, James Harden, Jayson Tatum, Jordan Poole, Keegan Murray, Klay Thompson, Lauri Markkanen, Luka Doncic, Malik Beasley, Michael Porter Jr., Mikal Bridges, Mike Conley, Paul George, Sam Hauser, Stephen Curry, Tim Hardaway Jr., Tyrese Haliburton, and Tyrese Maxey.

Full Dataset:
| Player Name         | Season   | Total Threes |
|---------------------|----------|--------------|
| Anthony Edwards     | 2023-24  | 189.6        |
| Bogdan Bogdanovic   | 2023-24  | 237.0        |
| Brandon Miller      | 2023-24  | 185.0        |
| Buddy Hield         | 2023-24  | 218.4        |
| CJ McCollum         | 2023

In [146]:
with open('temp_data/chat_output.txt', 'w') as f:
    f.write(response.content)